In [ ]:
#Libraray Import
import numpy as np
import pandas as pd
from sklearn.model_selection import (
    StratifiedKFold, LeaveOneOut, train_test_split, GridSearchCV, RandomizedSearchCV
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve,auc
)
from sklearn.preprocessing import StandardScaler, RobustScaler
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.neighbors import NearestNeighbors
from collections import Counter
import time
import shap

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving medical_data_clean.csv to medical_data_clean.csv


In [ ]:
#Reading the data
df = pd.read_csv('medical_data_clean.csv')
display(df.head())

,S1_T1,S2_T1,S3_T1,S4_T1,S5_T1,S6_T1,S1_T2,S2_T2,S3_T2,S4_T2,...,S5_T3,S6_T3,Age,Sex,Smoking,LC_stage,LC_type,Group,Group_name,lscm
0,3.180000e-07,3.730000e-07,0.000230,9.420000e-07,5.440000e-07,7.010000e-07,6.790000e-07,6.920000e-07,0.000202,0.000004,...,8.850000e-07,7.660000e-07,50,1,1,3,0,1,LC group,4
1,3.300000e-07,2.770000e-07,0.000268,1.120000e-06,8.950000e-07,1.050000e-06,2.660000e-07,1.890000e-07,0.000095,0.000002,...,1.160000e-06,9.310000e-07,68,0,0,4,1,1,LC group,3
2,5.450000e-07,4.810000e-07,0.000527,2.400000e-06,2.260000e-06,2.310000e-06,7.360000e-07,5.290000e-07,0.000266,0.000005,...,4.080000e-06,3.070000e-06,70,1,1,1,1,1,LC group,4
3,3.780000e-07,4.190000e-07,0.000241,1.100000e-06,9.490000e-07,1.010000e-06,3.580000e-07,3.750000e-07,0.000095,0.000002,...,1.420000e-06,1.070000e-06,66,0,0,4,1,1,LC group,3
4,4.260000e-07,4.780000e-07,0.000276,1.440000e-06,8.630000e-07,1.050000e-06,4.350000e-07,4.680000e-07,0.000120,0.000003,...,1.400000e-06,1.190000e-06,69,1,1,1,1,1,LC group,4


In [ ]:
#Defining the X and Y variable
X = df.drop(columns=["Group","LC_stage","LC_type","lscm","Group_name"])
y = df["Group"]
display(X.head())
display(y.head())

,S1_T1,S2_T1,S3_T1,S4_T1,S5_T1,S6_T1,S1_T2,S2_T2,S3_T2,S4_T2,...,S6_T2,S1_T3,S2_T3,S3_T3,S4_T3,S5_T3,S6_T3,Age,Sex,Smoking
0,3.180000e-07,3.730000e-07,0.000230,9.420000e-07,5.440000e-07,7.010000e-07,6.790000e-07,6.920000e-07,0.000202,0.000004,...,1.500000e-06,5.560000e-07,2.720000e-07,0.000035,0.000004,8.850000e-07,7.660000e-07,50,1,1
1,3.300000e-07,2.770000e-07,0.000268,1.120000e-06,8.950000e-07,1.050000e-06,2.660000e-07,1.890000e-07,0.000095,0.000002,...,8.760000e-07,2.110000e-07,1.750000e-07,0.000035,0.000004,1.160000e-06,9.310000e-07,68,0,0
2,5.450000e-07,4.810000e-07,0.000527,2.400000e-06,2.260000e-06,2.310000e-06,7.360000e-07,5.290000e-07,0.000266,0.000005,...,2.750000e-06,6.980000e-07,5.250000e-07,0.000119,0.000013,4.080000e-06,3.070000e-06,70,1,1
3,3.780000e-07,4.190000e-07,0.000241,1.100000e-06,9.490000e-07,1.010000e-06,3.580000e-07,3.750000e-07,0.000095,0.000002,...,9.480000e-07,2.760000e-07,3.300000e-07,0.000038,0.000005,1.420000e-06,1.070000e-06,66,0,0
4,4.260000e-07,4.780000e-07,0.000276,1.440000e-06,8.630000e-07,1.050000e-06,4.350000e-07,4.680000e-07,0.000120,0.000003,...,1.170000e-06,3.210000e-07,3.700000e-07,0.000044,0.000006,1.400000e-06,1.190000e-06,69,1,1


,Group
0,1
1,1
2,1
3,1
4,1


In [ ]:
# Performing Nested CV along SHAP
PARAM_DISTRIBUTIONS = {
    'n_estimators': [50, 75, 100],
    'max_depth': [2, 3, 4, 5],
    'learning_rate': [0.05, 0.1],
    'gamma': [0.0, 0.1],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'reg_lambda': [1, 1.5, 2],
}


def _fit_fold_model(X_train, y_train, n_inner_splits, random_state, use_smote):
    """Scale +SMOTE + inner RandomizedSearchCV, all fit on
    training data only. Returns the fitted scaler and the best fitted model."""
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    if use_smote:
        smote = SMOTE(random_state=random_state)
        X_train_scaled, y_train = smote.fit_resample(X_train_scaled, y_train)

    inner_skf = StratifiedKFold(n_splits=n_inner_splits, shuffle=True,
                                 random_state=random_state)
    base_model = XGBClassifier(random_state=random_state, eval_metric='logloss')
    search = RandomizedSearchCV(
        estimator=base_model,
        param_distributions=PARAM_DISTRIBUTIONS,
        n_iter=50,
        cv=inner_skf,
        scoring='roc_auc',
        n_jobs=-1,
        random_state=random_state,
        verbose=0,
        refit=True,
    )
    search.fit(X_train_scaled, y_train)
    return scaler, search.best_estimator_, search.best_params_

In [ ]:
def nested_cv_with_honest_shap(
    X, y,
    n_outer_splits=5,
    n_inner_splits=3,
    random_state=42,
    use_smote=True,
    top_k=10,
):

    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X)
    feature_names = list(X.columns)

    X_values = X.values
    y_values = y.values if isinstance(y, pd.Series) else np.asarray(y)

    outer_skf = StratifiedKFold(n_splits=n_outer_splits, shuffle=True,
                                 random_state=random_state)

    full_results = {'roc_auc': [], 'accuracy': [], 'sensitivity': [], 'specificity': []}
    topk_results = {'roc_auc': [], 'accuracy': [], 'sensitivity': [], 'specificity': []}

    # Out-of-fold SHAP storage: one row per original sample
    oof_shap = np.full((len(X), len(feature_names)), np.nan)
    topk_selection_counts = pd.Series(0, index=feature_names)

    total_start = time.time()

    for outer_fold, (train_idx, test_idx) in enumerate(
            outer_skf.split(X_values, y_values), 1):

        fold_start = time.time()
        print(f"\n{'─'*60}\nOUTER Fold {outer_fold}/{n_outer_splits}\n{'─'*60}")

        X_train_outer, X_test_outer = X_values[train_idx], X_values[test_idx]
        y_train_outer, y_test_outer = y_values[train_idx], y_values[test_idx]

        # ---- (A) Full-feature model for this fold -------------------------
        scaler, model, best_params = _fit_fold_model(
            X_train_outer, y_train_outer, n_inner_splits, random_state, use_smote)

        X_test_outer_scaled = scaler.transform(X_test_outer)
        y_pred = model.predict(X_test_outer_scaled)
        y_prob = model.predict_proba(X_test_outer_scaled)[:, 1]

        cm = confusion_matrix(y_test_outer, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        full_results['roc_auc'].append(roc_auc_score(y_test_outer, y_prob))
        full_results['accuracy'].append(accuracy_score(y_test_outer, y_pred))
        full_results['sensitivity'].append(tp / (tp + fn) if (tp + fn) else np.nan)
        full_results['specificity'].append(tn / (tn + fp) if (tn + fp) else np.nan)

        print(f"  [Full model]  ROC-AUC={full_results['roc_auc'][-1]:.4f}  "
              f"Acc={full_results['accuracy'][-1]:.4f}  params={best_params}")

        # ---- (B) Out-of-fold SHAP for this fold's test samples -------------
        # Background = this fold's (scaled, pre-SMOTE) outer-training data,

        X_train_outer_scaled_bg = scaler.transform(X_train_outer)
        explainer = shap.TreeExplainer(model, feature_perturbation="tree_path_dependent")
        sv_test = explainer.shap_values(X_test_outer_scaled)
        # xgboost binary classifier via TreeExplainer returns a single array
        # for the positive class in most shap versions; if it returns a list,
        # take index 1 (positive class).
        if isinstance(sv_test, list):
            sv_test = sv_test[1]
        oof_shap[test_idx, :] = sv_test

        # ---- (C) Nested top-k feature selection, fold-local ----------------
        # Rank features using |SHAP| computed on the OUTER-TRAIN data only

        sv_train = explainer.shap_values(X_train_outer_scaled_bg)
        if isinstance(sv_train, list):
            sv_train = sv_train[1]
        mean_abs_shap_train = np.abs(sv_train).mean(axis=0)
        ranking = pd.Series(mean_abs_shap_train, index=feature_names) \
                    .sort_values(ascending=False)
        fold_top_k = list(ranking.head(top_k).index)
        topk_selection_counts[fold_top_k] += 1
        print(f"  Fold {outer_fold} top-{top_k} features: {fold_top_k}")

        top_k_idx = [feature_names.index(f) for f in fold_top_k]
        X_train_topk = X_train_outer[:, top_k_idx]
        X_test_topk = X_test_outer[:, top_k_idx]

        scaler_k, model_k, best_params_k = _fit_fold_model(
            X_train_topk, y_train_outer, n_inner_splits, random_state, use_smote)
        X_test_topk_scaled = scaler_k.transform(X_test_topk)
        y_pred_k = model_k.predict(X_test_topk_scaled)
        y_prob_k = model_k.predict_proba(X_test_topk_scaled)[:, 1]

        cm_k = confusion_matrix(y_test_outer, y_pred_k, labels=[0, 1])
        tn_k, fp_k, fn_k, tp_k = cm_k.ravel()
        topk_results['roc_auc'].append(roc_auc_score(y_test_outer, y_prob_k))
        topk_results['accuracy'].append(accuracy_score(y_test_outer, y_pred_k))
        topk_results['sensitivity'].append(tp_k / (tp_k + fn_k) if (tp_k + fn_k) else np.nan)
        topk_results['specificity'].append(tn_k / (tn_k + fp_k) if (tn_k + fp_k) else np.nan)

        print(f"  [Top-{top_k} model]  ROC-AUC={topk_results['roc_auc'][-1]:.4f}  "
              f"Acc={topk_results['accuracy'][-1]:.4f}")
        print(f"  Fold time: {time.time() - fold_start:.1f}s")

    # ---- Aggregate ----------------------------------------------------------
    def _summ(d):
        return {k: (np.mean(v), np.std(v)) for k, v in d.items()}

    print(f"\n{'='*60}\nFINAL RESULTS (mean ± std across {n_outer_splits} outer folds)\n{'='*60}")
    for label, res in [("Full feature model", full_results),
                        (f"Nested top-{top_k} model", topk_results)]:
        s = _summ(res)
        print(f"\n{label}:")
        for metric, (m, sd) in s.items():
            print(f"  {metric:12s}: {m:.4f} ± {sd:.4f}")

    print(f"\nFeature selection stability (folds out of {n_outer_splits} in which "
          f"each feature appeared in the top-{top_k}):")
    print(topk_selection_counts.sort_values(ascending=False).to_string())

    print(f"\nTotal runtime: {time.time() - total_start:.1f}s")

    oof_shap_df = pd.DataFrame(oof_shap, columns=feature_names, index=X.index)

    return {
        'full_model': full_results,
        'topk_model': topk_results,
        'oof_shap': oof_shap_df,
        'topk_selection_counts': topk_selection_counts.sort_values(ascending=False),
    }

In [ ]:
 results = nested_cv_with_honest_shap(
        X, y,
        n_outer_splits=5,
        n_inner_splits=3,
        random_state=42,
        use_smote=True,
        top_k=10,
    )


────────────────────────────────────────────────────────────
OUTER Fold 1/5
────────────────────────────────────────────────────────────
  [Full model]  ROC-AUC=1.0000  Acc=0.9583  params={'subsample': 0.9, 'reg_lambda': 1.5, 'n_estimators': 50, 'max_depth': 2, 'learning_rate': 0.1, 'gamma': 0.0, 'colsample_bytree': 0.9}
  Fold 1 top-10 features: ['S4_T2', 'S4_T1', 'S4_T3', 'S5_T2', 'Smoking', 'Age', 'S6_T1', 'S5_T1', 'S2_T1', 'S3_T2']
  [Top-10 model]  ROC-AUC=1.0000  Acc=0.9583
  Fold time: 8.5s

────────────────────────────────────────────────────────────
OUTER Fold 2/5
────────────────────────────────────────────────────────────
  [Full model]  ROC-AUC=0.9720  Acc=0.8750  params={'subsample': 0.9, 'reg_lambda': 1.5, 'n_estimators': 50, 'max_depth': 4, 'learning_rate': 0.1, 'gamma': 0.0, 'colsample_bytree': 0.9}
  Fold 2 top-10 features: ['S4_T2', 'S4_T3', 'S4_T1', 'S2_T1', 'S5_T2', 'Age', 'S6_T1', 'S1_T1', 'S5_T1', 'S5_T3']
  [Top-10 model]  ROC-AUC=0.9720  Acc=0.9167
  Fold time:

In [ ]:
mean_abs_shap = results['oof_shap'].abs().mean(axis=0).sort_values(ascending=False)
print("\nOut-of-fold mean |SHAP| ranking (use this for Table 5):")
print(mean_abs_shap.head(15))


Out-of-fold mean |SHAP| ranking (use this for Table 5):
S4_T2      1.546985
S4_T1      0.912369
S4_T3      0.430530
Age        0.325605
Smoking    0.291999
S6_T1      0.202091
S5_T1      0.189758
S5_T2      0.178264
S2_T1      0.176896
S1_T1      0.168832
S3_T2      0.088430
S1_T3      0.078055
S2_T2      0.070729
S3_T3      0.051863
S2_T3      0.048721
dtype: float64


In [ ]:
oof_shap = results['oof_shap']            # DataFrame, n_samples x n_features
X_display = X.loc[oof_shap.index]         # matching original feature values, for coloring

# ---- Figure 5: SHAP summary / beeswarm plot --------------------------------
shap.summary_plot(oof_shap.values, X_display, max_display=15, show=False)
plt.tight_layout()
plt.savefig("Figure5_SHAP_summary_honest.png", dpi=300, bbox_inches="tight")
plt.close()

# ---- Figure 6: Waterfall plot for one example sample ------------------------

sample_idx = 0
expl = shap.Explanation(
    values=oof_shap.iloc[sample_idx].values,
    base_values=oof_shap.values.mean(),  # approximate baseline; see note below
    data=X_display.iloc[sample_idx].values,
    feature_names=list(X_display.columns),
)
shap.plots.waterfall(expl, show=False)
plt.tight_layout()
plt.savefig("Figure6_SHAP_waterfall_honest.png", dpi=300, bbox_inches="tight")
plt.close()

# ----Figure 7: Dependence plot (e.g. S4_T2 vs S6_T2) -------------------------
shap.dependence_plot(
    "S4_T2", oof_shap.values, X_display,
    interaction_index="S6_T2", show=False
)
plt.tight_layout()
plt.savefig("Figure7_SHAP_dependence_honest.png", dpi=300, bbox_inches="tight")
plt.close()

print("Saved Figure5_SHAP_summary_honest.png, Figure6_SHAP_waterfall_honest.png, "
      "Figure7_SHAP_dependence_honest.png")


Saved Figure5_SHAP_summary_honest.png, Figure6_SHAP_waterfall_honest.png, Figure7_SHAP_dependence_honest.png
